# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个问题（可技术、也可科普）
- **输出**：清晰、易懂的解释（Markdown）
- **对比路径**：同一问题分别问云端 `gpt-4o-mini`（流式）和本地 `llama3.2`（一次性）

这是你在课程期间自己也能天天用的工具。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | GPT 路径逐块 `print` |
| OpenAI 云端模型 | 常量 `MODEL_GPT = 'gpt-4o-mini'` |
| Ollama 本地模型 | OpenAI 兼容端点 `http://localhost:11434/v1`，模型 `llama3.2` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地路径需 Ollama 已启动并拉取 `llama3.2`
3. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮展示 Markdown 回答
from IPython.display import Markdown, display  # keep for nice formatting
# 从 openai 导入 OpenAI 客户端类：既可打云端，也可打 Ollama 的 OpenAI 兼容接口
from openai import OpenAI
# 导入 httpx：自定义 HTTP 客户端（本练习用 trust_env=False 忽略系统代理）
import httpx


In [ ]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境准备：加载密钥并创建 OpenAI 客户端 ==========

# 加载 .env：override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI API Key（名字必须是 OPENAI_API_KEY，与 .env 键一致）
api_key = os.getenv('OPENAI_API_KEY')

# ---------- 简单自检：帮你早点发现密钥问题（打印文案保持原样，便于对照报错） ----------
# 情况 1：根本没读到密钥
if not api_key:
    print("No API key was found!")
# 情况 2：格式不像常见的 sk-proj- 前缀（可能复制错了）
elif not api_key.startswith("sk-proj-"):
    print("Wrong API key format!")
# 情况 3：首尾有多余空格（复制粘贴常见坑）
elif api_key.strip() != api_key:
    print("API key has extra spaces!")
# 情况 4：看起来正常
else:
    print("API key found and looks good!")

# 创建默认 OpenAI 客户端：会自动从环境变量读取 OPENAI_API_KEY
openai = OpenAI()


In [ ]:
# ========== System Prompt：规定模型「以什么身份、什么风格」回答 ==========

# system_prompt 保留英文：这是发给模型的指令，翻译会改变回答行为
# 理念：system 定规则，user 放具体问题（下一格）
system_prompt = """You are a helpful assistant.
When given a question, provide a clear, 
simple and easy to understand explanation.
Use examples where helpful.
Format your response in Markdown."""


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 把问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的概念或一行代码，再分别跑下面 GPT / Llama 两格
question = """Can you help explain what is a black hole? 
How many black holes are there?"""


In [ ]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# 打印标题分隔线（展示用文案保持原样）
print("GPT-4o-mini Answer 🤖")
print("-" * 50)

# 组装 Chat Completions 的 messages：system 定风格，user 放问题
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

# 发起流式请求：stream=True 表示边生成边返回增量 delta
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    temperature=0.7,
    stream=True
)

# 遍历流式块：有内容就立刻打印，实现「打字机」效果
# end="" 不换行拼接；flush=True 立刻刷到屏幕
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)


In [ ]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）回答 ==========

# 打印标题分隔线（展示用文案保持原样）
print("Llama 3.2 Answer 🦙")
print("-" * 50)

# 创建指向本地 Ollama 的 OpenAI 兼容客户端
# base_url 必须是本机 Ollama 的 /v1；api_key 对本地通常任意非空即可
llama_client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',
    # trust_env=False：忽略系统代理，避免代理把 localhost 请求拐走
    http_client=httpx.Client(
        trust_env=False  # 👈 ignore system proxy settings
    )
)

# 同一套 messages：便于和 GPT 路径公平对比
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

# 非流式调用：等整段生成完再取 message.content
response = llama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
    temperature=0.7
)

# 用 Markdown 在笔记本里漂亮展示完整回答
display(Markdown(response.choices[0].message.content))
